In [ ]:
import ollama
import json
import re


# -----------------------------
# 1. CALL LLM
# -----------------------------
def call_llm(user_input):
    prompt = f"""
You are a system dynamics expert. Your job is to extract stock-and-flow structures from text descriptions.

Text:
\"\"\"{user_input}\"\"\"

REQUIRED OUTPUT FORMAT - You MUST return valid JSON with ALL of these keys:
{{
  "stocks": ["Name of accumulating quantity"],
  "inflows": ["Name of rate that increases stock"],
  "outflows": ["Name of rate that decreases stock"],
  "auxiliary": ["Name of helper variables"],
  "delays": [
    {{
      "name": "Name of the delayed variable",
      "input": "Variable being delayed",
      "delay_time": 3,
      "delay_type": "first_order",
      "initial_value": 0
    }}
  ],
  "equations": {{
    "stock": "INTEG(inflow_name - outflow_name, initial_value)",
    "inflow": "mathematical formula",
    "outflow": "mathematical formula or DELAY_VAR(input, delay_time, initial)",
    "auxiliary": "formula"
  }},
  "units": {{}}
}}

RULES:
1. STOCKS accumulate over time (e.g., Population, Water Level)
2. INFLOWS increase stocks (unit = stock/time)
3. OUTFLOWS decrease stocks (unit = stock/time)
4. AUXILIARY variables (helper variables that aren't stocks or flows)
   - constants
   - ratios
   - probabilities
   - intermediate calculations

5. DELAYS capture time-lagged effects between variables (STRICT RULE):
    A. Use "first_order" (SMOOTHI) when:
      - The delay represents perception, adjustment, or gradual response
      - Keywords: "gradually", "adjust", "perceive", "estimate", "smooth"
      - Example: demand perception, response time adjustment

   B. Use "third_order" (DELAY3) when:
      - The delay represents real-world processes with stages or pipelines
      - Keywords: "training", "onboarding", "processing", "multi-stage", "build-up"
      - Example: hiring → training → productive staff

   C. Use "pipeline" (DELAY FIXED) when:
      - The delay is a fixed transit or delivery time
      - Keywords: "after exactly", "shipping delay", "delivery time", "fixed lag"
      - Example: shipping takes exactly 5 days

   D. If the delay type is unclear, default to "third_order"

   E. Initial value rules:
      - If not specified, ALWAYS use 0 for flows
      - DO NOT leave initial value undefined

6. EQUATIONS:
   - Each stock MUST use INTEG(inflow - outflow, initial_value)
   - Delayed variables use SMOOTHI(), DELAY3(), or DELAY FIXED()
       - first_order: SMOOTHI(input_var, delay_time, initial_value)
       - third_order: DELAY3(input_var, delay_time)
       - pipeline: DELAY FIXED(input_var, delay_time, initial_value)
   - Use SIMPLE linear relationships unless clearly specified
   - Avoid complex nonlinear formulas unless necessary

7. UNITS (VERY IMPORTANT):
   - Every variable MUST have a unit
   - Stocks: base unit (e.g., people, houses, dollars)
   - Flows: stock/time (e.g., people/month)
   - Delays: time (e.g., month)
   - Probabilities: dimensionless
   - Rates: 1/time

8. NAMING:
   - Use clear, readable names (no abbreviations)
   - Be consistent across equations and units

9. Every variable name that appears in any equation MUST be declared in exactly one of: stocks, inflows, outflows, or auxiliary. If you write a variable in an equation, it MUST appear in the lists. No "ghost" variables allowed. [STRICT]
10. OUTPUT RULE: Return ONLY JSON. No explanation. No markdown.

EXAMPLE WITH DELAY:
Input: "New hires take 3 months to become productive. Hiring rate depends on the workforce gap."
Output:
{{
  "stocks": ["Workforce"],
  "inflows": ["Onboarding Completions"],
  "outflows": ["Attrition"],
  "auxiliary": ["Desired Workforce", "Workforce Gap", "Attrition Rate"],
  "delays": [
    {{
      "name": "Onboarding Completions",
      "input": "Hiring Rate",
      "delay_time": 3,
      "delay_type": "third_order",
      "initial_value": 0
    }}
  ],
  "equations": {{
    "Workforce": "INTEG(Onboarding Completions - Attrition, 100)",
    "Hiring Rate": "MAX(0, Workforce Gap / 1)",
    "Onboarding Completions": "DELAY3(Hiring Rate, 3, 0)",
    "Attrition": "Workforce * Attrition Rate",
    "Desired Workforce": "150",
    "Workforce Gap": "Desired Workforce - Workforce",
    "Attrition Rate": "0.05"
  }},
  "units": {{
    "Workforce": "People",
    "Onboarding Completions": "People/Month",
    "Attrition": "People/Month",
    "Hiring Rate": "People/Month",
    "Desired Workforce": "People",
    "Workforce Gap": "People",
    "Attrition Rate": "1/Month"
  }}
}}

EXAMPLE WITHOUT DELAY (for comparison):
Input: "When a post gets more views, people share it more, bringing in even more views."
Output:
{{
  "stocks": ["Post Views"],
  "inflows": ["New Views from Shares"],
  "outflows": ["View Decay"],
  "auxiliary": ["Share Probability", "Decay Rate"],
  "delays": [],
  "equations": {{
    "Post Views": "INTEG(New Views from Shares - View Decay, 0)",
    "New Views from Shares": "Post Views * Share Probability",
    "View Decay": "Post Views * Decay Rate",
    "Share Probability": "0.1",
    "Decay Rate": "0.05"
  }},
  "units": {{
    "Post Views": "Views",
    "New Views from Shares": "Views/Day",
    "View Decay": "Views/Day",
    "Share Probability": "Dimensionless",
    "Decay Rate": "1/Day"
  }}
}}

Now analyze the given text and return JSON only.
"""
    try:
        response = ollama.chat(
            model='qwen3.6',
            messages=[{"role": "user", "content": prompt}]
        )
        return response['message']['content']

    except Exception as e:
        print("LLM call failed:", e)
        return ""

# -----------------------------
# 2. PARSE JSON SAFELY
# -----------------------------
def parse_json(response_text):
    """Extract and parse JSON from various formats."""
    
    if not response_text or not isinstance(response_text, str):
        return None
    
    response_text = response_text.strip()
    
    # Strategy 1: Try direct parsing
    try:
        return json.loads(response_text)
    except json.JSONDecodeError:
        pass
    
    # Strategy 2: Extract from markdown code blocks
    try:
        json_match = re.search(r'```(?:json)?\s*(\{[\s\S]*?\})\s*```', 
                             response_text, re.DOTALL)
        if json_match:
            return json.loads(json_match.group(1))
    except json.JSONDecodeError:
        pass
    
    # Strategy 3: Extract using brace counting (most robust!)
    try:
        start = response_text.find('{')
        if start != -1:
            # ✅ Count braces instead of using greedy regex
            brace_count = 0
            end = start
            for i in range(start, len(response_text)):
                if response_text[i] == '{':
                    brace_count += 1
                elif response_text[i] == '}':
                    brace_count -= 1
                    if brace_count == 0:
                        end = i + 1
                        break
            
            if brace_count == 0:  # Found matching braces
                json_str = response_text[start:end]
                return json.loads(json_str)
    except json.JSONDecodeError as e:
        pass
    
    # ✅ Clear error message with sample of what failed
    print(f"❌ JSON parsing failed. Response:\n{response_text[:500]}")
    return None

# -----------------------------
# 3. VALIDATION
# -----------------------------
def validate(data):
    errors = []
    
    if not data:
        return ["No data parsed"]

    stocks = data.get("stocks", [])
    inflows = data.get("inflows", [])
    outflows = data.get("outflows", [])
    auxiliary = data.get("auxiliary", [])
    equations = data.get("equations", {})
    units = data.get("units", {})
    delays = data.get("delays", [])

    # --- Basic presence checks ---
    if not stocks:
        errors.append("Missing stocks")
    if not inflows:
        errors.append("No inflows defined")
    if not outflows:
        errors.append("No outflows defined")
    if not equations:
        errors.append("No equations provided")

    # --- All variables declared ---
    declared_vars = set(stocks + inflows + outflows + auxiliary)
    equation_vars = set(equations.keys())

    # Check all declared vars have equations
    missing_eq = declared_vars - equation_vars
    if missing_eq:
        errors.append(f"Missing equations for: {missing_eq}")

    # --- Check stock equation format ---
    for stock in stocks:
        eq = equations.get(stock, "")
        if "INTEG(" not in eq:
            errors.append(f"{stock} is not using INTEG")

    # --- Detect ghost variables ---
    def extract_vars(expr):
        return set(re.findall(r"[A-Za-z_][A-Za-z0-9_ ]*", expr))

    used_vars = set()
    for eq in equations.values():
        used_vars |= extract_vars(eq)

    ghost_vars = used_vars - declared_vars - {"INTEG", "MAX", "MIN", "DELAY3", "SMOOTH", "DELAY FIXED"}
    if ghost_vars:
        errors.append(f"Undeclared variables used: {ghost_vars}")

    # --- Units check ---
    for var in declared_vars:
        if var not in units:
            errors.append(f"Missing unit for: {var}")

    # --- Delay consistency check ---
    for d in delays:
        name = d["name"]
        dtype = d["delay_type"]
        eq = equations.get(name, "")

        if dtype == "third_order" and "DELAY3" not in eq:
            errors.append(f"{name} should use DELAY3")

        if dtype == "first_order" and "SMOOTH" not in eq:
            errors.append(f"{name} should use SMOOTH")

        if dtype == "pipeline" and "DELAY FIXED" not in eq:
            errors.append(f"{name} should use DELAY FIXED")

    return errors

# -----------------------------
# 4. GENERATE MDL
# -----------------------------
def generate_mdl(data):
    lines = []

    # Header
    lines.append("{UTF-8}")

    equations = data.get("equations", {})
    units = data.get("units", {})

    # -----------------------
    # Name normalization
    # -----------------------
    def normalize_name(name):
        return name.lower().replace(" ", "_")

    # Build mapping: original → normalized
    all_vars = list(equations.keys())
    name_map = {var: normalize_name(var) for var in all_vars}

    def format_unit(unit):
        """Convert to Vensim style (Month instead of month)"""
        if not unit:
            return "dimensionless"
        return unit.replace("month", "Month").replace("per", "/")
    # -----------------------
    # Format equations
    # -----------------------
    def format_equation(eq):
        eq = eq.strip()

        # Replace variable names with normalized names
        # Sort by length to avoid partial replacement issues
        sorted_vars = sorted(name_map.keys(), key=len, reverse=True)
        for var in sorted_vars:
            pattern = r"\b" + re.escape(var) + r"\b"
            eq = re.sub(pattern, name_map[var], eq)
        # Normalize to lowercase
        eq = eq.lower()

        # Restore Vensim keywords
        keywords = [
            "INTEG", "DELAY3", "SMOOTH", "DELAY FIXED",
            "MAX", "MIN", "IF THEN ELSE"
        ]

        for kw in keywords:
            eq = eq.replace(kw.lower(), kw)

        return eq
   
     # -----------------------
    # Generate variables
    # -----------------------
    for var, eq in equations.items():
        var_name = name_map[var]
        eq_formatted = format_equation(eq)
        unit = format_unit(units.get(var, "dimensionless"))

        lines.append(f"{var_name} = {eq_formatted}")
        lines.append(f"~ {unit}")
        lines.append(f"~ |")
        lines.append("")

        # Control section
    lines.append("********************************************************")
    lines.append(".Control")
    lines.append("********************************************************~")
    lines.append("Simulation Control Parameters")
    lines.append("|")
    lines.append("")
    lines.append("FINAL TIME  = 100")
    lines.append("~ Month")
    lines.append("~ The final time for the simulation.")
    lines.append("|")
    lines.append("")
    lines.append("INITIAL TIME  = 0")
    lines.append("~ Month")
    lines.append("~ The initial time for the simulation.")
    lines.append("|")
    lines.append("")
    lines.append("SAVEPER  =")
    lines.append("        TIME STEP")
    lines.append("~ Month")
    lines.append("~ The frequency with which output is stored.")
    lines.append("|")
    lines.append("")
    lines.append("TIME STEP  = 1")
    lines.append("~ Month")
    lines.append("~ The time step for the simulation.")
    lines.append("|")

    return "\n".join(lines)

# -----------------------------
# 5. FULL PIPELINE
# -----------------------------
def run_pipeline(user_input):
    print("🔹 Input:", user_input)
    print("\n" + "="*70)
 
    raw = call_llm(user_input)
    print("\n🔹 Raw LLM Output:")
    print(raw)
    print("="*70)
 
    parsed = parse_json(raw)
    print("\n🔹 Parsed JSON (before extraction):")
    if parsed:
        print(json.dumps(parsed, indent=2))
    else:
        print("Failed to parse JSON")
 
    errors = validate(parsed)
 
    if errors:
        print("\n❌ Validation Errors:")
        for error in errors:
            print(f"  - {error}")
        return None
 
    mdl = generate_mdl(parsed)
    print("\n✅ MDL Output:")
    print(mdl)
    print("="*70)
 
    return parsed, mdl

In [2]:
run_pipeline("An increase in complaints leads to an increase in staff count, " \
"as the organization hires more employees to handle the issues. " \
"As staff count increases, mentoring workload for senior staff also increases, since more new employees require guidance and training." \
" However, increased mentoring workload reduces senior availability (negative relationship), as senior staff have less time for other tasks." \
" Lower senior availability leads to longer response time, meaning it takes more time to address customer issues. " \
"As response time increases, complaints increase (positive relationship), completing the reinforcing loop. " \
"There is a delay between staff count and mentoring workload, representing the time it takes for new hires to require mentoring.")

🔹 Input: An increase in complaints leads to an increase in staff count, as the organization hires more employees to handle the issues. As staff count increases, mentoring workload for senior staff also increases, since more new employees require guidance and training. However, increased mentoring workload reduces senior availability (negative relationship), as senior staff have less time for other tasks. Lower senior availability leads to longer response time, meaning it takes more time to address customer issues. As response time increases, complaints increase (positive relationship), completing the reinforcing loop. There is a delay between staff count and mentoring workload, representing the time it takes for new hires to require mentoring.


🔹 Raw LLM Output:


🔹 Parsed JSON (before extraction):
Failed to parse JSON

❌ Validation Errors:
  - No data parsed
